# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Karthick-177/Flyrank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

The intended unit of analysis is daily performance for one content item and one client on one report date. The time window is one day, represented by report_date. The full table covers 2025-01-27 through 2026-06-30, and the verification query shows that the expected grain is not perfectly unique because a small number of duplicate grain combinations exist.

In [12]:
# Section 1 — Grain and date-window check

q1 = """
SELECT
    COUNT(*) AS rows_checked,
    COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS unique_grain_rows,
    MIN(report_date) AS first_report_date,
    MAX(report_date) AS last_report_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
"""

print("QUERY 1 — Grain and date-window check")
print(con.sql(q1).df())

QUERY 1 — Grain and date-window check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   rows_checked  unique_grain_rows first_report_date last_report_date
0      78835655           78829265        2025-01-27       2026-06-30


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: gsc_impressions, gsc_clicks, gsc_sum_position, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec, sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other, scroll_events.

Label: A future performance outcome derived from the same content/client daily performance history, defined using a future time window rather than current-day values.

Context: report_date, client_hash_id, content_hash_id, client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available.

Excluded: Any label-derived or future-period fields, because they would leak information from the outcome into the features. Client-identifying information is also excluded because the dataset is pseudonymized and the analysis should remain at the aggregate/decision-support level.

In [13]:
# Section 2 — Verify the fields used in the data contract

q2 = """
DESCRIBE
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
"""

print("SECTION 2 — Available fields")
print(con.sql(q2).df().to_string(index=False))

SECTION 2 — Available fields
             column_name column_type null  key default extra
             report_date        DATE  YES None    None  None
          client_hash_id     VARCHAR  YES None    None  None
         content_hash_id     VARCHAR  YES None    None  None
          client_has_gsc     BOOLEAN  YES None    None  None
          client_has_ga4     BOOLEAN  YES None    None  None
      gsc_data_available     BOOLEAN  YES None    None  None
      ga4_data_available     BOOLEAN  YES None    None  None
         gsc_impressions      BIGINT  YES None    None  None
              gsc_clicks      BIGINT  YES None    None  None
        gsc_sum_position      BIGINT  YES None    None  None
        gsc_avg_position      DOUBLE  YES None    None  None
           ga4_pageviews      BIGINT  YES None    None  None
            ga4_sessions      BIGINT  YES None    None  None
               ga4_users      BIGINT  YES None    None  None
    ga4_engaged_sessions      BIGINT  YES None    None  

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verification plan:

1. Verify the grain by checking whether each report_date + client_hash_id + content_hash_id combination is unique.
2. Check the number of rows and distinct clients/content items.
3. Check missing values in the key fields and data-availability fields.
4. Verify the date range covered by the table and confirm that report_date represents a daily window.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

rel = "hf://datasets/FlyRank/internship-warehouse"

# Query 1 — verify the grain
q1 = """
SELECT
    COUNT(*) AS rows_checked,
    COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id) AS unique_grain_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print("QUERY 1 — Grain check")
print(con.sql(q1).df())

# Query 2 — verify missing values
q2 = """
SELECT
    COUNT(*) AS rows_checked,
    COUNT(*) FILTER (WHERE report_date IS NULL) AS missing_report_date,
    COUNT(*) FILTER (WHERE client_hash_id IS NULL) AS missing_client_id,
    COUNT(*) FILTER (WHERE content_hash_id IS NULL) AS missing_content_id
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print("\nQUERY 2 — Missing-value check")
print(con.sql(q2).df())

# Query 3 — verify data availability
q3 = """
SELECT
    COUNT(*) AS rows_before_filter,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,
    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print("\nQUERY 3 — Availability check")
print(con.sql(q3).df())

QUERY 1 — Grain check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   rows_checked  unique_grain_rows
0       9841378            9841378

QUERY 2 — Missing-value check
   rows_checked  missing_report_date  missing_client_id  missing_content_id
0       9841378                    0                  0                   0

QUERY 3 — Availability check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   rows_before_filter  gsc_available_rows  ga4_available_rows
0             9841378             3611061              413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The data cannot fully represent clients or periods with unbalanced history, especially early periods where GSC data is unavailable. It also cannot provide a complete view when GSC or GA4 availability is missing. Future-window labels may overlap across rows, so results should not be interpreted as fully independent observations.

In [15]:
# Section 4 — Verify data availability limits

q4 = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
    ) AS both_available_rows,
    MIN(report_date) AS first_report_date,
    MAX(report_date) AS last_report_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
"""

print("SECTION 4 — Data availability limits")
print(con.sql(q4).df())

SECTION 4 — Data availability limits


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  gsc_available_rows  ga4_available_rows  both_available_rows  \
0    78835655            28970051             2816455              2453375   

  first_report_date last_report_date  
0        2025-01-27       2026-06-30  


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.